# Krishna River — Feature Engineering

**Input:** `krishna_master_dataset_clean.csv` (from `03_krishna_eda.ipynb`) + `krishna_station_coverage.csv`
**Output:** `krishna_features.csv` — a model-ready table with lag, rainfall, seasonal, and ENSO features

**Why these specific features** (established in the EDA notebook):
- Same-day rainfall is a weak predictor of same-day streamflow (correlation 0.07–0.61 across stations) — flow responds to rain with a delay, so the model needs *history*, not just today's values.
- Temperature tracks monsoon season rather than driving streamflow directly in this non-snowmelt basin — represented here via explicit month/monsoon flags, with raw `tmax` also kept so the model can use it if useful.
- ENSO (El Niño/La Niña) is a core focus of this project — represented via a **3-month rolling average of ONI**, since its effect on the Indian monsoon operates on a seasonal timescale, not a daily one.

## 0. Load cleaned data and the station eval list

In [1]:
import pandas as pd
import numpy as np

df_final = pd.read_csv("../data/processed/krishna_master_dataset_clean.csv", parse_dates=["date"])
station_coverage = pd.read_csv("../data/processed/krishna_station_coverage.csv", index_col="station_name")
eval_stations = station_coverage[station_coverage["usable_for_eval"]].index.tolist()

df_features = df_final[df_final["station_name"].isin(eval_stations)].copy()
df_features = df_features.sort_values(["station_name", "date"]).reset_index(drop=True)

print(f"Rows: {len(df_features)}, Stations: {df_features['station_name'].nunique()}")

Rows: 306966, Stations: 54


## 1. Streamflow lag features

Yesterday's flow (`streamflow_lag_1`) alone turned out to be a very strong predictor on its own (this is the persistence baseline used later) — including several lags lets the model use recent trend, not just the single most recent value.

In [2]:
for lag in [1, 2, 3, 7]:
    df_features[f"streamflow_lag_{lag}"] = (
        df_features.groupby("station_name")["streamflow"].shift(lag)
    )

## 2. Rainfall accumulation features

Since same-day rainfall barely correlates with same-day streamflow, rolling sums over the past few days capture the delayed, cumulative effect of rain on river flow instead.

In [3]:
for window in [3, 7, 14]:
    df_features[f"rainfall_sum_{window}d"] = (
        df_features.groupby("station_name")["rainfall"]
        .transform(lambda s: s.rolling(window, min_periods=1).sum())
    )

## 3. Seasonal encoding

Temperature's relationship with streamflow in the EDA looked like it was tracking monsoon season rather than driving flow directly (Krishna basin isn't snowmelt-fed). `month` and `is_monsoon` give the model a direct seasonal signal; raw `tmax` is kept too (Section 5) in case it carries information beyond what month alone captures.

In [4]:
df_features["month"] = df_features["date"].dt.month
df_features["is_monsoon"] = df_features["month"].isin([6, 7, 8, 9, 10]).astype(int)

## 4. ENSO feature — 3-month rolling ONI

ONI is naturally a slow-moving index (already a 3-month running average of sea surface temperature anomalies). Using the raw daily value as a feature tests the wrong timescale for what's actually a seasonal effect. A 90-day rolling average, computed per station so no lookback crosses a station boundary, better represents "what climate state was this basin in during this period" — which is the physically meaningful quantity.

The scientific analysis of *how much* this matters for Krishna basin streamflow is done separately in `05_krishna_split_and_baseline_model.ipynb`, Section 6 — this cell just prepares the feature.

In [5]:
df_features["oni_3month_avg"] = (
    df_features.groupby("station_name")["oni"]
    .transform(lambda s: s.rolling(90, min_periods=1).mean())
)

## 5. Preview and save

In [6]:
preview_cols = [
    "date", "station_name", "streamflow", "streamflow_lag_1",
    "rainfall_sum_7d", "is_monsoon", "tmax", "oni_3month_avg"
]
print(df_features[preview_cols].head(10))

        date station_name  streamflow  streamflow_lag_1  rainfall_sum_7d  \
0 2018-12-28       Airani       8.569               NaN         0.027975   
1 2018-12-29       Airani       7.021             8.569         0.162443   
2 2018-12-31       Airani       6.857             7.021         0.162443   
3 2019-01-01       Airani       6.034             6.857         0.162443   
4 2019-01-02       Airani       6.030             6.034         0.162443   
5 2019-01-03       Airani       7.206             6.030         0.162443   
6 2019-01-04       Airani       8.082             7.206         0.162443   
7 2019-01-05       Airani       5.638             8.082         0.134468   
8 2019-01-07       Airani       3.379             5.638         0.000000   
9 2019-01-08       Airani       3.843             3.379         0.000000   

   is_monsoon       tmax  oni_3month_avg  
0           0  28.743551        1.050000  
1           0  28.959879        1.050000  
2           0  28.921656        1.

In [7]:
df_features.to_csv("../data/processed/krishna_features.csv", index=False)
print("Saved:", df_features.shape)

Saved: (306966, 23)


## Summary

Feature table saved with:
- **Lags:** `streamflow_lag_1/2/3/7`
- **Rainfall accumulation:** `rainfall_sum_3d/7d/14d`
- **Season:** `month`, `is_monsoon`
- **Temperature:** `tmax` (raw, carried through from the source data)
- **ENSO:** `oni_3month_avg`

Next: `05_krishna_split_and_baseline_model.ipynb` — time-based split, baseline model, and the ENSO-streamflow analysis.